# LINet3 Gradient Diagnosis on SUN RGB-D

**Purpose:** Diagnose why LINet3's first-layer filters aren't learning, using real SUN RGB-D data.

Sections:
- Phase 1c: Data Scaling Check
- Phase 1d: Dead ReLU Probe
- Phase 3a: Gradient Cliff Test (per-layer gradient magnitudes)
- Phase 3b: Integration Weight Magnitude Analysis
- Phase 4a: Gradient Magnitude Over 50 Training Steps
- Phase 4b: Weight Change After 50 Steps
- Phase 4c: Standard ResNet-18 Baseline Comparison
- Diagnosis Summary

## 1. Environment Setup & GPU

In [ ]:
# Check GPU availability and specs
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

## 2. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 3. Clone Repository

In [ ]:
import os
from pathlib import Path

PROJECT_NAME = "Multi-Stream-Neural-Networks"
GITHUB_REPO = "https://github.com/clingergab/Multi-Stream-Neural-Networks.git"
REPO_DIR = f"/content/{PROJECT_NAME}"

os.chdir('/content')

if Path(REPO_DIR).exists() and Path(f"{REPO_DIR}/.git").exists():
    print(f"Repo already exists: {REPO_DIR}")
    os.chdir(REPO_DIR)
    !git pull
else:
    if Path(REPO_DIR).exists():
        !rm -rf {REPO_DIR}
    print(f"Cloning from {GITHUB_REPO}...")
    !git clone {GITHUB_REPO} {REPO_DIR}
    if not Path(REPO_DIR).exists():
        raise RuntimeError("Failed to clone repository")
    os.chdir(REPO_DIR)

print(f"Working directory: {os.getcwd()}")

## 4. Install Dependencies

In [ ]:
# Install required packages
!pip install -q thop

## 5. Copy Dataset to Local Disk

In [ ]:
from pathlib import Path
import shutil

DRIVE_DATASET_PATH = "/content/drive/MyDrive/datasets/sunrgbd_19_traintest"
LOCAL_DATASET_PATH = "/dev/shm/sunrgbd_19_traintest"

if not Path(LOCAL_DATASET_PATH).exists():
    print("Copying dataset to local disk for fast I/O...")
    shutil.copytree(DRIVE_DATASET_PATH, LOCAL_DATASET_PATH)
    print("Done!")
else:
    print(f"Dataset already at {LOCAL_DATASET_PATH}")

## 6. Imports & Setup

In [ ]:
import sys
sys.path.insert(0, REPO_DIR)

import torch
import torch.nn.functional as F
import numpy as np
from collections import defaultdict

from src.models.linear_integration.li_net3 import li_resnet18
from src.models.linear_integration.li_net3.conv import LIConv2d
from src.data_utils.sunrgbd_dataset import get_sunrgbd_dataloaders
from src.training.optimizers import create_stream_optimizer
from src.utils.seed import set_seed

SEED = 42
set_seed(SEED)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

## 7. Load Dataset

In [ ]:
train_loader, val_loader, test_loader = get_sunrgbd_dataloaders(
    data_root=LOCAL_DATASET_PATH,
    batch_size=64,
    num_workers=4,
    seed=SEED,
    normalize=True,
)
print(f"Train batches: {len(train_loader)}")
print(f"Test batches: {len(test_loader)}")

## Phase 1c: Data Scaling Check

**What we're looking for:**
- A healthy normalized tensor should have mean ~0.0, std ~1.0, min/max roughly in [-3, 3]
- If RGB mean is ~115 (stats computed on [0,255] but applied after /255.0 scaling), every value goes to ~-114 and the first ReLU kills everything
- If Depth values are raw uint16 millimeters (0-8000+), they'll overwhelm the optimizer
- We also inspect `norm_stats.json` directly to catch [0,255] vs [0,1] space mismatches

In [ ]:
# --- Step 1: Inspect norm_stats.json directly ---
import json as _json

norm_stats_path = os.path.join(LOCAL_DATASET_PATH, 'norm_stats.json')
with open(norm_stats_path) as f:
    norm_stats = _json.load(f)

print("=== norm_stats.json (raw values) ===")
print(f"  RGB mean:   {norm_stats['rgb_mean']}")
print(f"  RGB std:    {norm_stats['rgb_std']}")
print(f"  Depth mean: {norm_stats['depth_mean']}")
print(f"  Depth std:  {norm_stats['depth_std']}")

# Sanity check: RGB stats should be in [0, 1] range (computed on float/255.0)
# If rgb_mean values are >1.0, stats were computed on [0, 255] space!
rgb_mean_max = max(norm_stats['rgb_mean'])
if rgb_mean_max > 1.0:
    print(f"\n  CRITICAL: RGB mean {rgb_mean_max:.1f} > 1.0!")
    print("  Stats were computed on [0, 255] but dataloader divides by 255 BEFORE normalizing.")
    print("  This shifts ALL values to ~-{:.0f}, killing every ReLU!".format(rgb_mean_max))
else:
    print(f"\n  OK: RGB mean in [0, 1] range (computed on float/255.0 space)")

# Depth stats should be in meters (typically mean ~1-4m for indoor scenes)
depth_mean = norm_stats['depth_mean'][0]
if depth_mean > 100:
    print(f"\n  CRITICAL: Depth mean {depth_mean:.1f} >> 10!")
    print("  Stats may be in millimeters but dataloader converts to meters (/1000).")
elif depth_mean < 0.01:
    print(f"\n  WARNING: Depth mean {depth_mean:.4f} very small -- check units")
else:
    print(f"  OK: Depth mean {depth_mean:.2f}m (reasonable for indoor scenes)")


In [ ]:
# --- Step 2: Check actual tensor values from the dataloader ---
batch = next(iter(train_loader))
rgb_batch, depth_batch, labels = batch

print("=== RGB TENSOR (from dataloader) ===")
print(f"  Shape: {rgb_batch.shape}")
print(f"  Mean:  {rgb_batch.mean().item():.4f}")
print(f"  Std:   {rgb_batch.std().item():.4f}")
print(f"  Min:   {rgb_batch.min().item():.4f}")
print(f"  Max:   {rgb_batch.max().item():.4f}")

print("\n=== DEPTH TENSOR (from dataloader) ===")
print(f"  Shape: {depth_batch.shape}")
print(f"  Mean:  {depth_batch.mean().item():.4f}")
print(f"  Std:   {depth_batch.std().item():.4f}")
print(f"  Min:   {depth_batch.min().item():.4f}")
print(f"  Max:   {depth_batch.max().item():.4f}")

# --- Diagnose specific failure modes ---
print("\n=== FAILURE MODE CHECKS ===")

# Check 1: [0,255] vs [0,1] space shift
# If mean is far from 0 (e.g., -114), stats were computed in wrong space
rgb_mean = rgb_batch.mean().item()
if abs(rgb_mean) > 10:
    print(f"  CRITICAL: RGB mean = {rgb_mean:.1f} (far from 0!)")
    print("  Norm stats are in [0,255] space but applied after /255.0 scaling.")
    print("  Every value is massively negative -> first ReLU kills ALL neurons.")
elif abs(rgb_mean) > 1:
    print(f"  WARNING: RGB mean = {rgb_mean:.2f} (should be ~0 after normalization)")
else:
    print(f"  OK: RGB mean = {rgb_mean:.4f} (close to 0)")

# Check 2: Depth outliers (raw 16-bit values leaking through)
depth_max = depth_batch.max().item()
depth_min = depth_batch.min().item()
if depth_max > 100:
    print(f"  CRITICAL: Depth max = {depth_max:.1f} (raw mm values leaking through!)")
    print("  Depth is not being converted to meters before normalization.")
elif depth_max > 20:
    print(f"  WARNING: Depth max = {depth_max:.1f} (unusually large for normalized data)")
else:
    print(f"  OK: Depth range [{depth_min:.2f}, {depth_max:.2f}] (reasonable)")

# Check 3: Per-channel RGB stats (catch single-channel issues)
print("\n  Per-channel RGB stats:")
for ch, name in enumerate(['R', 'G', 'B']):
    ch_mean = rgb_batch[:, ch].mean().item()
    ch_std = rgb_batch[:, ch].std().item()
    print(f"    {name}: mean={ch_mean:.4f}, std={ch_std:.4f}")

# Check 4: Scale ratio between modalities
rgb_scale = rgb_batch.std().item()
depth_scale = depth_batch.std().item()
scale_ratio = max(rgb_scale, depth_scale) / min(rgb_scale, depth_scale)
print(f"\n  RGB/Depth scale ratio: {scale_ratio:.2f}x", end="")
if scale_ratio > 5:
    print(" -- WARNING: large mismatch, optimizer may favor one stream!")
else:
    print(" -- OK")

## 8. Create Model

In [ ]:
model = li_resnet18(
    num_classes=19,
    stream_input_channels=[3, 1],
    width_multiplier=0.75,
    dropout_p=0.515,
    device=DEVICE,
    use_amp=False,  # Disable AMP for accurate gradient diagnostics
)
model.train()
print(f"Model on: {DEVICE}")

## Phase 1d: Dead ReLU Probe

In [ ]:
# Hook to capture activations before AND after ReLU
# bn1 with apply_relu=True returns post-ReLU output; we also capture its input (post-conv1, pre-BN)
activation_stats = {}
pre_relu_stats = {}

def capture_bn1_output(module, input, output):
    """
    input:  (stream_outputs, integrated, blanked_mask, ...) from conv1 — pre-BN, pre-ReLU
    output: (stream_outputs, integrated) — post-BN, post-ReLU (apply_relu=True)
    """
    # Post-ReLU: check for dead neurons
    stream_outputs, integrated = output
    for i, s in enumerate(stream_outputs):
        dead_frac = (s == 0).float().mean().item()
        activation_stats[f'stream_{i}_post_relu'] = dead_frac
    dead_frac = (integrated == 0).float().mean().item()
    activation_stats['integrated_post_relu'] = dead_frac

    # Pre-BN/Pre-ReLU: check if activations are overwhelmingly negative
    # input[0] is the list of stream outputs from conv1 (before BN)
    pre_bn_streams = input[0]
    pre_bn_integrated = input[1]
    for i, s in enumerate(pre_bn_streams):
        neg_frac = (s < 0).float().mean().item()
        pre_relu_stats[f'stream_{i}_pre_bn_neg_frac'] = neg_frac
        pre_relu_stats[f'stream_{i}_pre_bn_mean'] = s.mean().item()
    if pre_bn_integrated is not None:
        neg_frac = (pre_bn_integrated < 0).float().mean().item()
        pre_relu_stats['integrated_pre_bn_neg_frac'] = neg_frac
        pre_relu_stats['integrated_pre_bn_mean'] = pre_bn_integrated.mean().item()

hook = model.bn1.register_forward_hook(capture_bn1_output)

# Run one real batch
batch = next(iter(train_loader))
rgb_batch, depth_batch, labels = batch
rgb_batch = rgb_batch.to(DEVICE)
depth_batch = depth_batch.to(DEVICE)

with torch.no_grad():
    _ = model([rgb_batch, depth_batch])

hook.remove()

print("=== Dead ReLU Probe ===")
print("\nPost-ReLU activation sparsity (after bn1 + ReLU):")
for name, dead_frac in activation_stats.items():
    status = "CRITICAL" if dead_frac > 0.8 else "WARNING" if dead_frac > 0.5 else "OK"
    print(f"  {name}: {dead_frac*100:.1f}% zeros [{status}]")

print("\nPre-BN activation distribution (conv1 output, before BN/ReLU):")
for name, val in pre_relu_stats.items():
    if 'neg_frac' in name:
        status = "CRITICAL" if val > 0.9 else "WARNING" if val > 0.7 else "OK"
        print(f"  {name}: {val*100:.1f}% negative [{status}]")
    else:
        print(f"  {name}: {val:.4f}")

if any(v > 0.8 for v in activation_stats.values()):
    print("\nDead neurons detected! Integration weights may be shifting")
    print("activations negative, killing gradients at the stem.")
    if any(v > 0.9 for k, v in pre_relu_stats.items() if 'neg_frac' in k):
        print("Pre-BN activations are overwhelmingly negative — BN can't rescue them.")

## Phase 3a: Gradient Cliff Test -- Per-Layer Gradient Magnitudes

In [ ]:
model.train()
model.zero_grad()

batch = next(iter(train_loader))
rgb_batch, depth_batch, labels = batch
rgb_batch = rgb_batch.to(DEVICE)
depth_batch = depth_batch.to(DEVICE)
labels = labels.to(DEVICE)

logits = model([rgb_batch, depth_batch])
loss = F.cross_entropy(logits, labels)
loss.backward()

# Build gradient table
layers = [
    ("conv1", model.conv1),
    ("layer1.0.conv1", model.layer1[0].conv1),
    ("layer1.1.conv1", model.layer1[1].conv1),
    ("layer2.0.conv1", model.layer2[0].conv1),
    ("layer3.0.conv1", model.layer3[0].conv1),
    ("layer4.0.conv1", model.layer4[0].conv1),
]

print("=== Gradient Cliff Test ===")
print(f"{'Layer':<20} | {'stream0 wt grad':>15} | {'stream1 wt grad':>15} | {'integ_from_str':>15} | {'integrated_wt':>15}")
print("-" * 95)

for name, conv in layers:
    s0 = conv.stream_weights[0].grad.abs().mean().item() if conv.stream_weights[0].grad is not None else 0
    s1 = conv.stream_weights[1].grad.abs().mean().item() if conv.stream_weights[1].grad is not None else 0
    ifs = conv.integration_from_streams[0].grad.abs().mean().item() if conv.integration_from_streams[0].grad is not None else 0
    if conv.integrated_weight.numel() > 0 and conv.integrated_weight.grad is not None:
        iw = conv.integrated_weight.grad.abs().mean().item()
        iw_str = f"{iw:.2e}"
    else:
        iw_str = "N/A"
    print(f"{name:<20} | {s0:>15.2e} | {s1:>15.2e} | {ifs:>15.2e} | {iw_str:>15}")

fc_grad = model.fc.weight.grad.abs().mean().item()
print(f"{'fc.weight':<20} | {fc_grad:>15.2e} | {'':>15} | {'':>15} | {'':>15}")

# Ratio analysis
conv1_grad = model.conv1.stream_weights[0].grad.abs().mean().item()
layer4_grad = model.layer4[0].conv1.stream_weights[0].grad.abs().mean().item()
if conv1_grad > 0:
    ratio = layer4_grad / conv1_grad
    print(f"\nlayer4/conv1 stream gradient ratio: {ratio:.1f}x")
    if ratio > 100:
        print("LARGE gradient gap between deep and shallow layers!")

## Phase 3b: Integration Weight Magnitude Analysis

In [ ]:
# Re-define layers (in case this cell is run independently of Phase 3a)
layers = [
    ("conv1", model.conv1),
    ("layer1.0.conv1", model.layer1[0].conv1),
    ("layer1.1.conv1", model.layer1[1].conv1),
    ("layer2.0.conv1", model.layer2[0].conv1),
    ("layer3.0.conv1", model.layer3[0].conv1),
    ("layer4.0.conv1", model.layer4[0].conv1),
]

print("=== Integration Weight Magnitudes ===")
print(f"{'Layer':<20} | {'integrated_wt mean':>18} | {'integrated_wt std':>18} | {'shape':>20} | {'init type':>12}")
print("-" * 100)

for name, conv in layers:
    w = conv.integrated_weight
    if w.numel() > 0:
        in_ch = w.shape[1]
        out_ch = w.shape[0]
        init_type = "orthogonal" if in_ch == out_ch else "kaiming*0.1"
        print(f"{name:<20} | {w.abs().mean().item():>18.4e} | {w.std().item():>18.4e} | {str(tuple(w.shape)):>20} | {init_type:>12}")
    else:
        print(f"{name:<20} | {'N/A (empty)':>18} | {'N/A':>18} | {str(tuple(w.shape)):>20} | {'N/A':>12}")

print("\nIntegration from streams weights:")
print(f"{'Layer':<20} | {'stream0 weight':>15} | {'stream1 weight':>15}")
print("-" * 55)
for name, conv in layers:
    w0 = conv.integration_from_streams[0].abs().mean().item()
    w1 = conv.integration_from_streams[1].abs().mean().item()
    print(f"{name:<20} | {w0:>15.4e} | {w1:>15.4e}")

## Phase 4a: Gradient Magnitude Over 50 Training Steps

In [ ]:
import matplotlib.pyplot as plt

# Fresh model for training dynamics
set_seed(SEED)
model = li_resnet18(
    num_classes=19,
    stream_input_channels=[3, 1],
    width_multiplier=0.75,
    dropout_p=0.515,
    device=DEVICE,
    use_amp=False,
)
model.train()

optimizer = create_stream_optimizer(
    model,
    optimizer_type='adamw',
    stream_lrs=[7.22e-05, 1.59e-04],
    stream_weight_decays=[6.33e-05, 4.25e-05],
    shared_lr=1.63e-04,
    integration_weight_decay=1.30e-04,
)

# Save initial weights for Phase 4b comparison
initial_weights = {}
for name, param in model.named_parameters():
    if any(k in name for k in ['conv1.stream_weights', 'layer4', 'fc.weight']):
        initial_weights[name] = param.data.clone().cpu()

# Track gradients over 50 steps
grad_history = defaultdict(list)
train_iter = iter(train_loader)
num_steps = 50

for step in range(num_steps):
    try:
        batch = next(train_iter)
    except StopIteration:
        train_iter = iter(train_loader)
        batch = next(train_iter)

    rgb, depth, labels = batch
    rgb, depth, labels = rgb.to(DEVICE), depth.to(DEVICE), labels.to(DEVICE)

    optimizer.zero_grad()
    logits = model([rgb, depth])
    loss = F.cross_entropy(logits, labels)
    loss.backward()

    # Record gradient magnitudes — streams, integration, and classifier
    grad_history['conv1_stream0'].append(model.conv1.stream_weights[0].grad.abs().mean().item())
    grad_history['conv1_stream1'].append(model.conv1.stream_weights[1].grad.abs().mean().item())
    grad_history['conv1_integ_from_str'].append(model.conv1.integration_from_streams[0].grad.abs().mean().item())
    grad_history['layer4_stream0'].append(model.layer4[0].conv1.stream_weights[0].grad.abs().mean().item())
    grad_history['layer4_integ_from_str'].append(model.layer4[0].conv1.integration_from_streams[0].grad.abs().mean().item())
    grad_history['fc'].append(model.fc.weight.grad.abs().mean().item())
    grad_history['loss'].append(loss.item())

    optimizer.step()

    if (step + 1) % 10 == 0:
        print(f"Step {step+1}/{num_steps} | loss: {loss.item():.4f} | "
              f"conv1_s0 grad: {grad_history['conv1_stream0'][-1]:.2e} | "
              f"conv1_integ: {grad_history['conv1_integ_from_str'][-1]:.2e} | "
              f"fc grad: {grad_history['fc'][-1]:.2e}")

# Plot
fig, axes = plt.subplots(3, 1, figsize=(12, 12))

ax = axes[0]
ax.semilogy(grad_history['conv1_stream0'], label='conv1 stream0 (RGB)', alpha=0.8)
ax.semilogy(grad_history['conv1_stream1'], label='conv1 stream1 (Depth)', alpha=0.8)
ax.semilogy(grad_history['layer4_stream0'], label='layer4 stream0', alpha=0.8)
ax.semilogy(grad_history['fc'], label='fc', alpha=0.8)
ax.set_ylabel('Gradient magnitude (log scale)')
ax.set_xlabel('Training step')
ax.set_title('Stream Weight Gradient Magnitudes')
ax.legend()
ax.grid(True, alpha=0.3)

ax = axes[1]
ax.semilogy(grad_history['conv1_integ_from_str'], label='conv1 integ_from_streams', alpha=0.8)
ax.semilogy(grad_history['layer4_integ_from_str'], label='layer4 integ_from_streams', alpha=0.8)
ax.set_ylabel('Gradient magnitude (log scale)')
ax.set_xlabel('Training step')
ax.set_title('Integration Weight Gradient Magnitudes (the 1x1 mixer bottleneck)')
ax.legend()
ax.grid(True, alpha=0.3)

ax = axes[2]
ax.plot(grad_history['loss'], label='CE Loss', color='red')
ax.set_ylabel('Loss')
ax.set_xlabel('Training step')
ax.set_title('Training Loss')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Summary statistics
print("\n=== Gradient Summary ===")
for key in ['conv1_stream0', 'conv1_stream1', 'conv1_integ_from_str',
            'layer4_stream0', 'layer4_integ_from_str', 'fc']:
    vals = grad_history[key]
    print(f"  {key}: mean={np.mean(vals):.2e}, min={np.min(vals):.2e}, max={np.max(vals):.2e}")

conv1_mean = np.mean(grad_history['conv1_stream0'])
fc_mean = np.mean(grad_history['fc'])
if conv1_mean > 0:
    print(f"\n  fc/conv1 gradient ratio: {fc_mean/conv1_mean:.1f}x")
if conv1_mean < 1e-7:
    print("\n  VANISHING GRADIENTS at conv1! Mean gradient < 1e-7")

# Check integration bottleneck
conv1_integ_mean = np.mean(grad_history['conv1_integ_from_str'])
layer4_integ_mean = np.mean(grad_history['layer4_integ_from_str'])
print(f"\n  conv1 integration_from_streams grad: {conv1_integ_mean:.2e}")
print(f"  layer4 integration_from_streams grad: {layer4_integ_mean:.2e}")
if layer4_integ_mean > 0:
    print(f"  layer4/conv1 integration grad ratio: {layer4_integ_mean/conv1_integ_mean:.1f}x")

## Phase 4b: Weight Change After 50 Steps

In [ ]:
import torch.nn as nn

# Learning rates from the optimizer config (used to compute update-to-weight ratio)
STREAM_LRS = [7.22e-05, 1.59e-04]
SHARED_LR = 1.63e-04

print("=== Weight Change After 50 Training Steps ===")
print(f"{'Parameter':<50} | {'||delta||/||init||':>18} | {'update/weight':>14} | {'verdict':>10}")
print("-" * 100)

param_dict = dict(model.named_parameters())

for name, init_w in sorted(initial_weights.items()):
    current_w = param_dict[name].data.cpu()
    delta = (current_w - init_w).norm().item()
    init_norm = init_w.norm().item()
    if init_norm > 0:
        change_ratio = delta / init_norm
    else:
        change_ratio = float('inf') if delta > 0 else 0.0

    # Compute the update-to-weight ratio: how big is one optimizer step vs the weight scale?
    # update_size ≈ grad * lr;  weight_scale ≈ init_w.std()
    grad = param_dict[name].grad
    if grad is not None:
        mean_grad = grad.abs().mean().item()
        # Pick the right LR for this parameter group
        if 'stream_weights.0' in name or 'stream_biases.0' in name:
            lr = STREAM_LRS[0]
        elif 'stream_weights.1' in name or 'stream_biases.1' in name:
            lr = STREAM_LRS[1]
        else:
            lr = SHARED_LR
        estimated_step = mean_grad * lr
        weight_std = init_w.std().item() + 1e-8
        update_ratio = estimated_step / weight_std
        update_str = f"{update_ratio:.2e}"
    else:
        update_str = "no grad"

    verdict = "FROZEN" if change_ratio < 1e-5 else "SLOW" if change_ratio < 1e-3 else "OK"
    print(f"{name:<50} | {change_ratio:>18.6e} | {update_str:>14} | {verdict:>10}")

# Summary interpretation
print("\n=== Update-to-Weight Ratio Interpretation ===")
print("  > 1e-2: Healthy — weights escape initialization quickly")
print("  1e-3 to 1e-4: Slow — might need thousands of steps to learn features")
print("  < 1e-5: Effectively frozen — LR is too low for this weight scale")
print("  < 1e-6: Mathematically invalid — would need tens of thousands of epochs")

## Phase 4c: Standard ResNet-18 Baseline Comparison

In [ ]:
import torchvision.models as tv_models

# Create standard ResNet-18 for comparison (untrained, like LINet3 at step 0)
baseline = tv_models.resnet18(weights=None, num_classes=19)

# Match LINet3's actual training config: dropout_p=0.515 before classifier
# Dropout kills ~52% of backward-flowing gradients, so this matters for fair comparison
baseline.fc = nn.Sequential(
    nn.Dropout(p=0.515),
    nn.Linear(baseline.fc.in_features, 19),
)

baseline = baseline.to(DEVICE)
baseline.train()

# Single forward+backward on the same data distribution
baseline_optimizer = torch.optim.AdamW(baseline.parameters(), lr=7.22e-05)

batch = next(iter(train_loader))
rgb, depth, labels = batch
rgb, labels = rgb.to(DEVICE), labels.to(DEVICE)

baseline_optimizer.zero_grad()
logits = baseline(rgb)
loss = F.cross_entropy(logits, labels)
loss.backward()

baseline_conv1_grad = baseline.conv1.weight.grad.abs().mean().item()
baseline_layer4_grad = baseline.layer4[0].conv1.weight.grad.abs().mean().item()
baseline_fc_grad = baseline.fc[1].weight.grad.abs().mean().item()  # fc[1] is Linear after Dropout

# Compare: use INITIAL LINet3 gradients (step 0) for fair apples-to-apples comparison
linet_conv1_initial = grad_history['conv1_stream0'][0]
linet_layer4_initial = grad_history['layer4_stream0'][0]
linet_fc_initial = grad_history['fc'][0]

# Also show final gradients (step 49) to see if things changed
linet_conv1_final = grad_history['conv1_stream0'][-1]
linet_layer4_final = grad_history['layer4_stream0'][-1]
linet_fc_final = grad_history['fc'][-1]

print("=== Standard ResNet-18 vs LINet3 Gradient Comparison ===")
print("(Both untrained, both with dropout_p=0.515 before classifier)")
print(f"{'Metric':<20} | {'ResNet-18':>12} | {'LINet3 step0':>13} | {'Ratio':>8} | {'LINet3 step49':>14}")
print("-" * 80)

rows = [
    ('conv1 grad', baseline_conv1_grad, linet_conv1_initial, linet_conv1_final),
    ('layer4 grad', baseline_layer4_grad, linet_layer4_initial, linet_layer4_final),
    ('fc grad', baseline_fc_grad, linet_fc_initial, linet_fc_final),
]

for name, base, linet_init, linet_fin in rows:
    ratio = base / linet_init if linet_init > 0 else float('inf')
    print(f"{name:<20} | {base:>12.2e} | {linet_init:>13.2e} | {ratio:>8.1f}x | {linet_fin:>14.2e}")

if baseline_conv1_grad / max(linet_conv1_initial, 1e-20) > 100:
    print("\nLINet3 conv1 gradients are >100x smaller than standard ResNet!")
    print("This confirms the issue is LINet3-specific gradient attenuation.")
elif baseline_conv1_grad / max(linet_conv1_initial, 1e-20) > 10:
    print("\nLINet3 conv1 gradients are 10-100x smaller than standard ResNet.")
    print("Moderate gradient attenuation — may contribute to slow stem learning.")
else:
    print("\nLINet3 conv1 gradients are comparable to standard ResNet.")
    print("Gradient magnitude is NOT the issue — look at signal quality instead.")

## Diagnosis Summary

In [ ]:
print("=" * 80)
print("DIAGNOSIS SUMMARY")
print("=" * 80)

findings = []

# 1c: Norm stats space mismatch (from cell 16)
if rgb_mean_max > 1.0:
    findings.append(f"NORM STATS MISMATCH: RGB mean {rgb_mean_max:.1f} > 1.0 -- stats computed on [0,255] but applied on [0,1]!")
if depth_mean > 100:
    findings.append(f"DEPTH STATS MISMATCH: Depth mean {depth_mean:.1f} -- stats in mm but dataloader converts to meters!")

# 1c: Data scaling from actual tensors (from cell 17)
if abs(rgb_mean) > 10:
    findings.append(f"RGB SPACE SHIFT: RGB mean = {rgb_mean:.1f} -- all neurons dead after first ReLU!")
if depth_max > 100:
    findings.append(f"DEPTH OVERFLOW: Depth max = {depth_max:.1f} -- raw 16-bit values leaking through!")
if scale_ratio > 5:
    findings.append(f"SCALE MISMATCH: RGB/Depth scale ratio is {scale_ratio:.1f}x -- gradient imbalance at stem!")

# 1d: Dead ReLU (from cell 21)
for name, dead_frac in activation_stats.items():
    if dead_frac > 0.8:
        findings.append(f"DEAD RELU: {name} has {dead_frac*100:.1f}% dead neurons!")

# Vanishing gradients (from cell 27)
if conv1_mean < 1e-7:
    findings.append(f"VANISHING GRADIENTS: conv1 mean gradient = {conv1_mean:.2e}")

# Integration bottleneck (from cell 27)
if conv1_integ_mean < 1e-10:
    findings.append(f"INTEGRATION BOTTLENECK: conv1 integration_from_streams grad = {conv1_integ_mean:.2e}")

# Update-to-weight ratio (from cell 29)
# Check conv1 stream_weights specifically
conv1_s0_param = param_dict.get('conv1.stream_weights.0')
if conv1_s0_param is not None and conv1_s0_param.grad is not None:
    conv1_grad_mag = conv1_s0_param.grad.abs().mean().item()
    conv1_init_std = initial_weights.get('conv1.stream_weights.0', torch.zeros(1)).std().item() + 1e-8
    conv1_update_ratio = (conv1_grad_mag * STREAM_LRS[0]) / conv1_init_std
    if conv1_update_ratio < 1e-5:
        findings.append(
            f"OPTIMIZER PHYSICS: conv1 update/weight ratio = {conv1_update_ratio:.2e} "
            f"-- LR {STREAM_LRS[0]:.2e} is too small for Kaiming-scale weights"
        )

# Baseline comparison (from cell 31)
ratio_vs_baseline = baseline_conv1_grad / max(linet_conv1_initial, 1e-20)
if ratio_vs_baseline > 100:
    findings.append(f"LINET3-SPECIFIC: conv1 gradients {ratio_vs_baseline:.0f}x smaller than standard ResNet")

if findings:
    print("\nISSUES FOUND:")
    for i, f in enumerate(findings, 1):
        print(f"  {i}. {f}")
else:
    print("\nNo obvious issues detected. Further investigation needed.")

print("\n" + "=" * 80)